# OmniASR Patois Benchmark — Gradio Demo
Tests Meta's `omnilingual-asr` (fairseq2) for Jamaican Creole (Patois) ASR.

**Before running:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
#@title 1. Install dependencies (run once, ~3 min) — auto-restarts runtime when done
import subprocess, sys

subprocess.run(['apt-get', 'install', '-y', '-qq', 'libsndfile1', 'ffmpeg'], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.8.0', 'torchaudio==2.8.0', 'torchvision',
    '--index-url', 'https://download.pytorch.org/whl/cu128', '--upgrade'
], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', 'fairseq2',
    '--extra-index-url', 'https://fair.pkg.atmeta.com/fairseq2/whl/pt2.8.0/cu128'
], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'omnilingual-asr', 'silero-vad', 'jiwer', 'gradio', 'transformers'
], check=True)

# MUST be last + force-reinstall so C extensions match Python files
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--upgrade', '--force-reinstall', 'numpy>=2.0,<2.3'
], check=True)

print('Install done. Restarting runtime so new numpy C extensions load...')
print('After restart: run cells 2 and 3 together — do NOT re-run cell 1.')
import os
os.kill(os.getpid(), 9)

In [ ]:
#@title 2. Setup helpers + model loaders
import torch, torchaudio, torchaudio.functional as TAF, time, tempfile, wave, os
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'none'
print(f'Device: {DEVICE}  |  GPU: {gpu_name}')

# ── shared audio helpers ──────────────────────────────────────────────────────

def load_audio(path, sr=16000):
    waveform, orig_sr = torchaudio.load(path)
    if orig_sr != sr:
        waveform = TAF.resample(waveform, orig_sr, sr)
    return waveform.mean(0).numpy()

def write_wav(chunk, sr=16000):
    tf = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
    tf.close()
    pcm = (chunk * 32767).clip(-32768, 32767).astype(np.int16)
    with wave.open(tf.name, 'wb') as wf:
        wf.setnchannels(1); wf.setsampwidth(2)
        wf.setframerate(sr); wf.writeframes(pcm.tobytes())
    return tf.name

def vad_chunk(audio, sr=16000, max_sec=30):
    max_samples = max_sec * sr
    if len(audio) <= max_samples:
        return [audio]
    try:
        from silero_vad import get_speech_timestamps, load_silero_vad
        vad = load_silero_vad()
        ts = get_speech_timestamps(torch.from_numpy(audio).float(), vad, sampling_rate=sr)
        if ts:
            chunks, start, last_end = [], 0, 0
            for seg in ts:
                if seg['end'] - start > max_samples and last_end > start:
                    chunks.append(audio[start:last_end])
                    start = seg['start']
                last_end = seg['end']
            chunks.append(audio[start:])
            return [c for c in chunks if len(c) > 0]
    except Exception:
        pass
    return [audio[i:i+max_samples] for i in range(0, len(audio), max_samples)]

VIDEO_EXTS = {'.mp4', '.mov', '.webm', '.mkv', '.avi', '.m4v', '.3gp'}

def extract_audio_from_video(video_path):
    import subprocess
    tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
    tmp.close()
    subprocess.run(
        ['ffmpeg', '-y', '-i', video_path, '-vn', '-ac', '1', '-ar', '16000', '-f', 'wav', tmp.name],
        check=True, capture_output=True
    )
    return tmp.name

def prep_audio(audio_path):
    """Load audio from file or video, return (numpy array, tmp_path_to_clean or None)."""
    ext = os.path.splitext(audio_path)[1].lower()
    tmp = None
    if ext in VIDEO_EXTS:
        audio_path = extract_audio_from_video(audio_path)
        tmp = audio_path
    return load_audio(audio_path), tmp

def score(text, reference):
    if not reference.strip():
        return ''
    try:
        from jiwer import wer, cer
        return f'WER: {wer(reference, text):.1%}  |  CER: {cer(reference, text):.1%}'
    except Exception as e:
        return f'Scoring error: {e}'

# ── omniASR ───────────────────────────────────────────────────────────────────

_omni_cache = {}

def _init_gang(device_str):
    from fairseq2.gang import create_fake_gangs, _thread_local
    _thread_local.current_gangs = [create_fake_gangs(torch.device(device_str))]

def get_omni_pipeline(model_card):
    if model_card not in _omni_cache:
        from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
        _init_gang(DEVICE)
        print(f'Loading omniASR {model_card} on {DEVICE}...')
        t0 = time.perf_counter()
        _omni_cache[model_card] = ASRInferencePipeline(model_card=model_card, device=DEVICE)
        print(f'Loaded in {(time.perf_counter()-t0)*1000:.0f} ms')
    return _omni_cache[model_card]

def run_omni(audio_path, lang, model_card, batch_size, reference):
    _init_gang(DEVICE)
    audio, tmp_vid = prep_audio(audio_path)
    audio_sec = len(audio) / 16000

    t_pre = time.perf_counter()
    chunks = vad_chunk(audio)
    tmp_paths = [write_wav(c) for c in chunks]
    prep_ms = int((time.perf_counter() - t_pre) * 1000)

    pipeline = get_omni_pipeline(model_card)

    # pipeline.transcribe() silently drops results after the first when given multiple paths —
    # call once per chunk to get complete output
    t_inf = time.perf_counter()
    results = []
    for path in tmp_paths:
        results.extend(list(pipeline.transcribe([path], lang=[lang], batch_size=1)))
    inf_ms = int((time.perf_counter() - t_inf) * 1000)

    for p in tmp_paths:
        os.unlink(p)
    if tmp_vid:
        os.unlink(tmp_vid)

    text = ' '.join(r.strip() for r in results if r.strip())
    rtf = round(inf_ms / 1000 / audio_sec, 3) if audio_sec > 0 else '-'
    stats = (f'Audio: {audio_sec:.1f}s | Chunks: {len(chunks)} | '
             f'Prep: {prep_ms}ms | Inference: {inf_ms}ms | RTF: {rtf}x')
    return text, stats, score(text, reference)

# ── MMS ───────────────────────────────────────────────────────────────────────

_mms_cache = {}

# Map LANGS codes (BCP-47 style) to MMS adapter codes
MMS_LANG_MAP = {
    'jam_Latn': 'jam',
    'eng_Latn': 'eng',
    'spa_Latn': 'spa',
    'fra_Latn': 'fra',
    'hat_Latn': 'hat',
}

def get_mms():
    if 'model' not in _mms_cache:
        from transformers import AutoProcessor, Wav2Vec2ForCTC
        print(f'Loading facebook/mms-1b-l1107 on {DEVICE}...')
        t0 = time.perf_counter()
        processor = AutoProcessor.from_pretrained('facebook/mms-1b-l1107')
        model = Wav2Vec2ForCTC.from_pretrained('facebook/mms-1b-l1107').to(DEVICE).eval()
        _mms_cache['model'] = model
        _mms_cache['processor'] = processor
        print(f'Loaded in {(time.perf_counter()-t0)*1000:.0f} ms')
    return _mms_cache['processor'], _mms_cache['model']

def run_mms(audio_path, lang, reference):
    mms_lang = MMS_LANG_MAP.get(lang, lang.split('_')[0])
    audio, tmp_vid = prep_audio(audio_path)
    audio_sec = len(audio) / 16000

    processor, model = get_mms()
    processor.tokenizer.set_target_lang(mms_lang)
    model.load_adapter(mms_lang)

    chunk_size = 25 * 16000
    chunks = [audio[i:i+chunk_size] for i in range(0, len(audio), chunk_size)]

    t_inf = time.perf_counter()
    texts = []
    for chunk in chunks:
        inputs = processor(chunk, sampling_rate=16000, return_tensors='pt')
        with torch.no_grad():
            logits = model(inputs.input_values.to(DEVICE)).logits
        ids = torch.argmax(logits, dim=-1)
        texts.append(processor.decode(ids[0]))
    inf_ms = int((time.perf_counter() - t_inf) * 1000)

    if tmp_vid:
        os.unlink(tmp_vid)

    text = ' '.join(t.strip() for t in texts if t.strip())
    rtf = round(inf_ms / 1000 / audio_sec, 3) if audio_sec > 0 else '-'
    stats = (f'Audio: {audio_sec:.1f}s | Chunks: {len(chunks)} | '
             f'Inference: {inf_ms}ms | RTF: {rtf}x')
    return text, stats, score(text, reference)

print('Ready')

In [ ]:
#@title 3. Launch Gradio UI
import gradio as gr

LANGS = [
    ('Jamaican Creole (Patois) — jam_Latn', 'jam_Latn'),
    ('English — eng_Latn', 'eng_Latn'),
    ('Spanish — spa_Latn', 'spa_Latn'),
    ('French — fra_Latn', 'fra_Latn'),
    ('Haitian Creole — hat_Latn', 'hat_Latn'),
    ('Portuguese — por_Latn', 'por_Latn'),
]
MODELS = [
    ('MMS 1B — facebook/mms-1b-l1107 (best Patois, ~2.4 GB)', 'mms|mms-1b-l1107'),
    ('omniASR CTC 300M  (faster, ~1.2 GB)',                    'omni|omniASR_CTC_300M'),
    ('omniASR CTC 1B    (more accurate, ~3.7 GB)',             'omni|omniASR_CTC_1B'),
]

def _to_path(v):
    if v is None: return None
    if isinstance(v, str): return v
    if isinstance(v, dict): return v.get('name') or v.get('path')
    return str(v)

def predict(audio_file, mic_audio, lang_label, model_label, batch_size, reference):
    src = _to_path(audio_file) or _to_path(mic_audio)
    if not src:
        return '', 'No audio provided.', '', None
    lang = dict(LANGS)[lang_label]
    model_key = dict(MODELS)[model_label]
    backend, model_card = model_key.split('|', 1)
    try:
        if backend == 'mms':
            text, stats, acc = run_mms(src, lang, reference or '')
        else:
            text, stats, acc = run_omni(src, lang, model_card, int(batch_size), reference or '')
        return text, stats, acc, src
    except Exception as e:
        import traceback
        return '', f'Error: {e}', traceback.format_exc(), None

with gr.Blocks(title='ASR Patois Benchmark', theme=gr.themes.Soft()) as demo:
    gr.Markdown('## ASR Patois Benchmark — MMS vs omniASR')
    gr.Markdown(
        'Upload **audio or video** (MP4, WAV, MP3, M4A, OGG, MOV, WEBM…) or record from mic.\n\n'
        '> **Microphone not working?** Open the `gradio.live` share URL printed below directly '
        'in a new browser tab — the embedded Colab iframe blocks mic access.\n\n'
        '> **First run per model** downloads weights and may take 1–2 min — button appears frozen, that\'s normal.'
    )

    with gr.Row():
        with gr.Column():
            audio_file = gr.File(
                file_types=['.mp4', '.mov', '.webm', '.mkv', '.avi', '.m4v',
                            '.mp3', '.wav', '.m4a', '.ogg', '.flac', '.aac', '.3gp'],
                label='Upload audio or video file'
            )
            mic_audio = gr.Audio(
                sources=['microphone'],
                type='filepath',
                label='Or record from microphone (open share URL in new tab)'
            )
            with gr.Row():
                lang_dd = gr.Dropdown(
                    choices=[l[0] for l in LANGS],
                    value='Jamaican Creole (Patois) — jam_Latn',
                    label='Language'
                )
                model_dd = gr.Dropdown(
                    choices=[m[0] for m in MODELS],
                    value='MMS 1B — facebook/mms-1b-l1107 (best Patois, ~2.4 GB)',
                    label='Model'
                )
            batch_sl = gr.Slider(1, 8, value=4, step=1, label='Batch size (omniASR only)')
            reference_in = gr.Textbox(
                lines=3,
                placeholder='Paste ground-truth transcript here to get WER / CER (optional)…',
                label='Reference transcript (optional)'
            )
            run_btn = gr.Button('Transcribe', variant='primary', size='lg')

        with gr.Column():
            transcript_out = gr.Textbox(label='Transcript', lines=20)
            stats_out = gr.Textbox(label='Timing', lines=2)
            accuracy_out = gr.Textbox(label='WER / CER', lines=1)
            audio_out = gr.Audio(label='Download recording', type='filepath')

    run_btn.click(
        predict,
        inputs=[audio_file, mic_audio, lang_dd, model_dd, batch_sl, reference_in],
        outputs=[transcript_out, stats_out, accuracy_out, audio_out]
    )

result = demo.launch(share=True, debug=True)
share_url = result[2] if isinstance(result, (list, tuple)) and len(result) > 2 else None
print('\n' + '='*60)
print('PASTE THIS INTO WELLNEST TRIAGE LAB → Gradio endpoint URL:')
print(share_url or '(no share URL — Gradio tunnelling unavailable)')
print('\nFor MICROPHONE: open the URL above directly in a new browser tab')
print('='*60)